# PAUDC Patois Dialogue — LoRA Fine-Tune (Kaggle free GPU)

Fine-tunes an open Qwen/Llama base on the project's curated Patois JSONL
(`pipeline/dialogue/patois_dataset.jsonl` — upload it as a Kaggle dataset).
Settings → Accelerator → **GPU T4**. Only worth running past ~2-5k curated
lines; below that the style adapter (`STYLE_ADAPTER.md`) wins on quality/effort.

**Rules:** curated original lines only; the authenticity rules from
`PAUDC_Dialogue_Voice.md` govern the dataset; outputs are drafts for human curation.

In [ ]:
%pip -q install transformers peft trl datasets bitsandbytes accelerate
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
from datasets import load_dataset
DATA = '/kaggle/input/paudc-patois/patois_dataset.jsonl'  # your uploaded dataset path
ds = load_dataset('json', data_files=DATA, split='train')
def fmt(r):
    return {'text': f"<|user|>\n{r['input']} (region: {r.get('region','any')}, tone: {r.get('tone','casual')})\n<|assistant|>\n{r['output']}"}
ds = ds.map(fmt); print(ds[0]['text'])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'  # small enough for free T4 with 4-bit
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj'])
cfg = SFTConfig(output_dir='/kaggle/working/patois-lora', per_device_train_batch_size=4,
                gradient_accumulation_steps=4, num_train_epochs=3, learning_rate=2e-4,
                logging_steps=10, save_strategy='epoch', max_length=256, packing=False)
trainer = SFTTrainer(model=model, train_dataset=ds, peft_config=lora, args=cfg)
trainer.train()
trainer.save_model('/kaggle/working/patois-lora/final')
print('adapter saved — download from the Output panel')

In [ ]:
# smoke-test the adapter
from peft import PeftModel
m = PeftModel.from_pretrained(model, '/kaggle/working/patois-lora/final')
p = '<|user|>\nNPC: ferry captain greeting passengers (region: coast, tone: humorous)\n<|assistant|>\n'
ids = tok(p, return_tensors='pt').to(m.device)
out = m.generate(**ids, max_new_tokens=60, do_sample=True, temperature=0.8)
print(tok.decode(out[0], skip_special_tokens=True))